# 12 — Ensemble

Combines the three strongest single models in their best configurations:
1. **MLP + FIFA** — notebook 09 best 3-class classifier (LL 1.0007, acc 54.7%)
2. **Regression-DC** — notebook 10 best goal model (LL 1.0151, acc 54.3%)
3. **XGBoost + FIFA** — notebook 09 alternative non-linear classifier (LL 1.0079, acc 52.3%)

Two combination strategies:
- **Simple average** of probability vectors (no extra parameters to overfit)
- **Tuned weights** via grid search on a single held-out WC (e.g., tune on WC 2018 only, then test on WC 2022)

Walk-forward backtest on WC 2010-2022.

In [1]:
import sys
sys.path.append('..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from itertools import product
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
import xgboost as xgb
from src.elo import EloModel
from src.draw_model import match_outcome
from src.regression_dc import RegressionDixonColesModel

## Build features (same as notebook 09 + notebook 10)

In [2]:
df_all = pd.read_csv('../data/processed/matches_competitive.csv', parse_dates=['date']).dropna(subset=['home_score','away_score'])
df_sv  = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])
elo = EloModel()
elo_enriched = elo.fit(df_all)
df = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                  on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])

HOME_ADVANTAGE=100; FLOOR=1e5
df['neutral'] = df['neutral'].fillna(False)
df['eff_elo_diff']  = (df['home_elo_pre'] + (~df['neutral']).astype(int)*HOME_ADVANTAGE) - df['away_elo_pre']
df['elo_diff_norm'] = df['eff_elo_diff']/400.0
df['log_value_diff']    = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR)/df['away_top_n_value_eur'].clip(lower=FLOOR))
df['outfield_age_diff'] = df['home_outfield_mean_age'] - df['away_outfield_mean_age']
df['top1_share_diff']   = df['home_top1_share'] - df['away_top1_share']
df['fifa_points_diff']  = (df['home_fifa_points'].fillna(0) - df['away_fifa_points'].fillna(0))/1000.0
# Reg-DC per-team features
df['home_elo']  = df['home_elo_pre']/100.0;  df['away_elo']  = df['away_elo_pre']/100.0
df['home_logv'] = np.log10(df['home_top_n_value_eur'].clip(lower=FLOOR))
df['away_logv'] = np.log10(df['away_top_n_value_eur'].clip(lower=FLOOR))
df['home_fpts'] = df['home_fifa_points'].fillna(0)/1000.0
df['away_fpts'] = df['away_fifa_points'].fillna(0)/1000.0
df['outcome']   = df.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)

FEAT_CLF = ['elo_diff_norm','log_value_diff','outfield_age_diff','top1_share_diff','fifa_points_diff']
RDC_FEAT = ['elo','logv','fpts']
needed = FEAT_CLF + [f'{s}_{c}' for s in ('home','away') for c in RDC_FEAT] + ['outcome']
valid = df.dropna(subset=needed)
valid = valid[(valid['home_top_n_value_eur']>FLOOR)&(valid['away_top_n_value_eur']>FLOOR)].copy()
print(f'Valid matches: {len(valid):,}')

Valid matches: 7,641


## Generate per-model predictions on each WC, then ensemble

In [3]:
def score(y_true, p):
    p = np.clip(p, 1e-6, 1-1e-6)
    oh = np.zeros_like(p); oh[np.arange(len(y_true)), y_true] = 1
    return {
        'log_loss': log_loss(y_true, p, labels=[0,1,2]),
        'accuracy': float((p.argmax(axis=1)==y_true).mean()),
        'brier':    float(np.mean(np.sum((p-oh)**2, axis=1))),
    }

def fit_xgb(X, y):
    return xgb.XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=300,
                              max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                              reg_lambda=1.0, eval_metric='mlogloss', n_jobs=4, verbosity=0).fit(X, y)
def fit_mlp(Xs, y):
    return MLPClassifier(hidden_layer_sizes=(32,16), activation='relu', max_iter=500,
                          learning_rate_init=0.005, alpha=1e-3, early_stopping=True, random_state=0).fit(Xs, y)
def proba_ord(clf, X):
    p = clf.predict_proba(X); order = [list(clf.classes_).index(c) for c in (0,1,2)]
    return p[:, order]

preds_per_year = {}
for year in [2010, 2014, 2018, 2022]:
    train = valid[valid['date'].dt.year < year]
    test  = valid[(valid['date'].dt.year == year) & (valid['tournament']=='FIFA World Cup')]
    if test.empty: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    Xtr = train[FEAT_CLF].to_numpy(); Xte = test[FEAT_CLF].to_numpy()
    sc = StandardScaler().fit(Xtr)
    
    p_mlp = proba_ord(fit_mlp(sc.transform(Xtr), y_train), sc.transform(Xte))
    p_xgb = proba_ord(fit_xgb(Xtr, y_train), Xte)
    
    rdc = RegressionDixonColesModel(RDC_FEAT, xi=0.00038).fit(train)
    p_rdc_df = rdc.predict_many(test)
    p_rdc = p_rdc_df[['p_away_win','p_draw','p_home_win']].to_numpy()
    
    preds_per_year[year] = {'y': y_test, 'mlp': p_mlp, 'xgb': p_xgb, 'rdc': p_rdc, 'n': len(test)}
    print(f'{year} done (n_test={len(test)}, n_train={len(train)})')

2010 done (n_test=64, n_train=1478)


2014 done (n_test=64, n_train=2818)


2018 done (n_test=64, n_train=4185)


2022 done (n_test=64, n_train=5779)


## Individual model scores + simple average

In [4]:
summary = []
for name in ['mlp', 'xgb', 'rdc']:
    yall = np.concatenate([d['y'] for d in preds_per_year.values()])
    pall = np.vstack([d[name] for d in preds_per_year.values()])
    s = score(yall, pall)
    summary.append({'model': name, **s})

# Simple average
for combos in [('mlp','xgb'), ('mlp','rdc'), ('xgb','rdc'), ('mlp','xgb','rdc')]:
    pall = []
    yall = []
    for d in preds_per_year.values():
        avg = np.mean([d[m] for m in combos], axis=0)
        avg = avg / avg.sum(axis=1, keepdims=True)
        pall.append(avg); yall.append(d['y'])
    yall = np.concatenate(yall); pall = np.vstack(pall)
    s = score(yall, pall)
    summary.append({'model': 'avg(' + '+'.join(combos) + ')', **s})

print(pd.DataFrame(summary).round(4).to_string(index=False))

           model  log_loss  accuracy  brier
             mlp    1.0111    0.5391 0.6015
             xgb    1.0025    0.5273 0.5951
             rdc    1.0149    0.5430 0.6053
    avg(mlp+xgb)    0.9957    0.5430 0.5915
    avg(mlp+rdc)    1.0086    0.5430 0.6007
    avg(xgb+rdc)    0.9957    0.5234 0.5919
avg(mlp+xgb+rdc)    0.9970    0.5469 0.5927


## Tuned weights: grid search on WC 2018, evaluate on WC 2022

Grid over (w_mlp, w_xgb, w_rdc) summing to 1, in increments of 0.1. Pick weights with best WC-2018 log loss, then apply to WC 2022.

In [5]:
def weighted_avg(d, w):
    p = w[0]*d['mlp'] + w[1]*d['xgb'] + w[2]*d['rdc']
    return p / p.sum(axis=1, keepdims=True)

weights_grid = [(i/10, j/10, (10-i-j)/10) for i in range(11) for j in range(11-i)]

tune = preds_per_year[2018]
tuned = []
for w in weights_grid:
    p = weighted_avg(tune, w)
    tuned.append({'w': w, 'll': score(tune['y'], p)['log_loss']})
tuned_df = pd.DataFrame(tuned).sort_values('ll')
print('Best WC-2018 tuning weights (top 5):')
print(tuned_df.head(5).round(4).to_string(index=False))

best_w = tuned_df.iloc[0]['w']
test = preds_per_year[2022]
p_test = weighted_avg(test, best_w)
s_test = score(test['y'], p_test)
s_eq   = score(test['y'], weighted_avg(test, (1/3, 1/3, 1/3)))
s_mlp  = score(test['y'], test['mlp'])
print(f'\nWC 2022 hold-out:')
print(f'  MLP alone           LL={s_mlp["log_loss"]:.4f}  acc={s_mlp["accuracy"]:.4f}')
print(f'  Equal-weight ens.   LL={s_eq["log_loss"]:.4f}  acc={s_eq["accuracy"]:.4f}')
print(f'  Tuned weights {best_w}  LL={s_test["log_loss"]:.4f}  acc={s_test["accuracy"]:.4f}')

Best WC-2018 tuning weights (top 5):
              w     ll
(1.0, 0.0, 0.0) 0.9749
(0.9, 0.0, 0.1) 0.9761
(0.9, 0.1, 0.0) 0.9762
(0.8, 0.0, 0.2) 0.9775
(0.8, 0.1, 0.1) 0.9775

WC 2022 hold-out:
  MLP alone           LL=1.0698  acc=0.5000
  Equal-weight ens.   LL=1.0573  acc=0.5312
  Tuned weights (1.0, 0.0, 0.0)  LL=1.0698  acc=0.5000


## CatBoost + walk-forward stacking

- Add CatBoost as a 4th base model (similar gradient boosting, different default calibration).
- **Stacking**: for each test WC, train a logistic-regression meta-learner on out-of-fold base predictions from *earlier* WCs. Walk-forward, no leakage.
- Compare against equal-weight average (best ensemble from above).

In [6]:
from catboost import CatBoostClassifier

def fit_cb(X, y):
    return CatBoostClassifier(iterations=400, depth=4, learning_rate=0.05,
                                loss_function='MultiClass', verbose=False,
                                random_seed=0, thread_count=4).fit(X, y)

# Regenerate preds with CatBoost added
preds_per_year = {}
for year in [2010, 2014, 2018, 2022]:
    train = valid[valid['date'].dt.year < year]
    test  = valid[(valid['date'].dt.year == year) & (valid['tournament']=='FIFA World Cup')]
    if test.empty: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    Xtr = train[FEAT_CLF].to_numpy(); Xte = test[FEAT_CLF].to_numpy()
    sc = StandardScaler().fit(Xtr)
    p_mlp = proba_ord(fit_mlp(sc.transform(Xtr), y_train), sc.transform(Xte))
    p_xgb = proba_ord(fit_xgb(Xtr, y_train), Xte)
    p_cb  = proba_ord(fit_cb(Xtr, y_train), Xte)
    rdc = RegressionDixonColesModel(RDC_FEAT, xi=0.00038).fit(train)
    p_rdc = rdc.predict_many(test)[['p_away_win','p_draw','p_home_win']].to_numpy()
    preds_per_year[year] = {'y': y_test, 'mlp': p_mlp, 'xgb': p_xgb, 'cb': p_cb, 'rdc': p_rdc, 'n': len(test)}
    print(f'{year} done')

# Individual scores including cb
rows = []
for name in ['mlp','xgb','cb','rdc']:
    y = np.concatenate([d['y'] for d in preds_per_year.values()])
    p = np.vstack([d[name] for d in preds_per_year.values()])
    s = score(y, p); rows.append({'model': name, **s})

# Equal-weight averages of various subsets
for combo in [('mlp','xgb','cb'), ('mlp','xgb','cb','rdc'), ('mlp','cb','rdc')]:
    ys = []; ps = []
    for d in preds_per_year.values():
        avg = np.mean([d[m] for m in combo], axis=0)
        avg = avg / avg.sum(axis=1, keepdims=True)
        ps.append(avg); ys.append(d['y'])
    s = score(np.concatenate(ys), np.vstack(ps))
    rows.append({'model': 'avg('+'+'.join(combo)+')', **s})
print(pd.DataFrame(rows).round(4).to_string(index=False))

2010 done


2014 done


2018 done


2022 done
              model  log_loss  accuracy  brier
                mlp    1.0111    0.5391 0.6015
                xgb    1.0025    0.5273 0.5951
                 cb    0.9949    0.5352 0.5918
                rdc    1.0149    0.5430 0.6053
    avg(mlp+xgb+cb)    0.9937    0.5391 0.5906
avg(mlp+xgb+cb+rdc)    0.9946    0.5391 0.5913
    avg(mlp+cb+rdc)    0.9991    0.5508 0.5945


## Walk-forward stacking

For each test WC year Y, the meta-learner (logit on base preds) is trained on every prior WC's predictions. No future leakage. WC 2010 can't be stacked (no prior data) — use equal-weight average there.

In [7]:
BASE_MODELS = ['mlp','xgb','cb','rdc']
stack_rows = []
for test_year in [2014, 2018, 2022]:
    # Meta-train: concat all preds from prior WCs
    prior_years = [y for y in preds_per_year if y < test_year]
    meta_X = np.hstack([np.vstack([preds_per_year[y][m] for y in prior_years]) for m in BASE_MODELS])
    meta_y = np.concatenate([preds_per_year[y]['y'] for y in prior_years])
    meta = LogisticRegression(max_iter=2000, C=1.0).fit(meta_X, meta_y)
    # Apply to test WC
    d = preds_per_year[test_year]
    test_X = np.hstack([d[m] for m in BASE_MODELS])
    p_stack = proba_ord(meta, test_X)
    p_eq    = np.mean([d[m] for m in BASE_MODELS], axis=0)
    p_eq    = p_eq / p_eq.sum(axis=1, keepdims=True)
    s_stack = score(d['y'], p_stack); s_eq = score(d['y'], p_eq)
    stack_rows.append({'year': test_year, 'n_meta_train': len(meta_y),
                       'stack_ll': s_stack['log_loss'], 'stack_acc': s_stack['accuracy'],
                       'eq_ll':    s_eq['log_loss'],    'eq_acc':    s_eq['accuracy']})
stack_df = pd.DataFrame(stack_rows)
print(stack_df.round(4).to_string(index=False))
print()
print(f"Walk-forward avg (2014-2022 only, n=192):")
print(f"  Equal-weight   LL={stack_df['eq_ll'].mean():.4f}  acc={stack_df['eq_acc'].mean():.4f}")
print(f"  Stacked logit  LL={stack_df['stack_ll'].mean():.4f}  acc={stack_df['stack_acc'].mean():.4f}")

 year  n_meta_train  stack_ll  stack_acc  eq_ll  eq_acc
 2014            64    0.9621     0.5938 0.9557  0.5938
 2018           128    0.9985     0.5625 0.9917  0.5312
 2022           192    1.0497     0.5312 1.0579  0.5312

Walk-forward avg (2014-2022 only, n=192):
  Equal-weight   LL=1.0018  acc=0.5521
  Stacked logit  LL=1.0034  acc=0.5625


## Adding big5_share_diff to the base feature set

In [8]:
# Reload with big5 features
df_sv  = pd.read_csv('../data/processed/matches_with_squad_value.csv', parse_dates=['date'])
df2 = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                   on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])
df2['neutral'] = df2['neutral'].fillna(False)
df2['eff_elo_diff']  = (df2['home_elo_pre'] + (~df2['neutral']).astype(int)*HOME_ADVANTAGE) - df2['away_elo_pre']
df2['elo_diff_norm'] = df2['eff_elo_diff']/400.0
df2['log_value_diff']    = np.log10(df2['home_top_n_value_eur'].clip(lower=FLOOR)/df2['away_top_n_value_eur'].clip(lower=FLOOR))
df2['outfield_age_diff'] = df2['home_outfield_mean_age'] - df2['away_outfield_mean_age']
df2['top1_share_diff']   = df2['home_top1_share'] - df2['away_top1_share']
df2['fifa_points_diff']  = (df2['home_fifa_points'].fillna(0) - df2['away_fifa_points'].fillna(0))/1000.0
# big5 features (NaN on non-WC matches → fill 0 = noop, but only WC test rows will use this)
df2['big5_share_diff'] = df2['home_big5_share_top_n'].fillna(0) - df2['away_big5_share_top_n'].fillna(0)
df2['outcome']   = df2.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)
FEAT_CLF_B5 = FEAT_CLF + ['big5_share_diff']
needed = FEAT_CLF_B5 + ['outcome']
valid2 = df2.dropna(subset=needed)
valid2 = valid2[(valid2['home_top_n_value_eur']>FLOOR)&(valid2['away_top_n_value_eur']>FLOOR)].copy()
print(f'Valid: {len(valid2):,}, with non-zero big5 diff: {(valid2["big5_share_diff"]!=0).sum()}')

Valid: 7,641, with non-zero big5 diff: 253


In [9]:
# Re-fit models with big5 added
preds_b5 = {}
for year in [2010, 2014, 2018, 2022]:
    train = valid2[valid2['date'].dt.year < year]
    test  = valid2[(valid2['date'].dt.year == year) & (valid2['tournament']=='FIFA World Cup')]
    if test.empty: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    Xtr = train[FEAT_CLF_B5].to_numpy(); Xte = test[FEAT_CLF_B5].to_numpy()
    sc = StandardScaler().fit(Xtr)
    p_mlp = proba_ord(fit_mlp(sc.transform(Xtr), y_train), sc.transform(Xte))
    p_xgb = proba_ord(fit_xgb(Xtr, y_train), Xte)
    p_cb  = proba_ord(fit_cb(Xtr, y_train), Xte)
    preds_b5[year] = {'y': y_test, 'mlp': p_mlp, 'xgb': p_xgb, 'cb': p_cb}

rows = []
for name in ['mlp','xgb','cb']:
    y = np.concatenate([d['y'] for d in preds_b5.values()])
    p = np.vstack([d[name] for d in preds_b5.values()])
    s = score(y, p); rows.append({'model': name+'_b5', **s})
for combo in [('mlp','xgb','cb'),]:
    ys = []; ps = []
    for d in preds_b5.values():
        avg = np.mean([d[m] for m in combo], axis=0)
        ps.append(avg / avg.sum(axis=1, keepdims=True)); ys.append(d['y'])
    s = score(np.concatenate(ys), np.vstack(ps))
    rows.append({'model': 'avg(' + '+'.join(combo) + ')_b5', **s})
print('With +big5_share_diff:')
print(pd.DataFrame(rows).round(4).to_string(index=False))

With +big5_share_diff:
             model  log_loss  accuracy  brier
            mlp_b5    1.0720    0.5078 0.6220
            xgb_b5    1.0237    0.5039 0.6043
             cb_b5    1.0019    0.5352 0.5935
avg(mlp+xgb+cb)_b5    1.0112    0.5234 0.5969


In [10]:
# WC-only training: every training row has big5 data
rows = []
for year in [2014, 2018, 2022]:
    train = valid2[(valid2['tournament']=='FIFA World Cup') & (valid2['date'].dt.year < year) & (valid2['date'].dt.year >= 2010)]
    test  = valid2[(valid2['tournament']=='FIFA World Cup') & (valid2['date'].dt.year == year)]
    if test.empty or len(train) < 30: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    row = {'year': year, 'n_train': len(train)}
    for feat_name, cols in [('no_b5', FEAT_CLF), ('+b5', FEAT_CLF_B5)]:
        for model_name, clf in [
            ('logit', LogisticRegression(max_iter=2000).fit(train[cols].to_numpy(), y_train)),
            ('cb',    fit_cb(train[cols].to_numpy(), y_train)),
        ]:
            p = proba_ord(clf, test[cols].to_numpy())
            s = score(y_test, p)
            row[f'{model_name}_{feat_name}_ll']  = s['log_loss']
            row[f'{model_name}_{feat_name}_acc'] = s['accuracy']
    rows.append(row)
r = pd.DataFrame(rows)
print(r.round(4).to_string(index=False))
print()
for n in ['logit_no_b5','logit_+b5','cb_no_b5','cb_+b5']:
    print(f'  {n:14s} LL={r[f"{n}_ll"].mean():.4f}  acc={r[f"{n}_acc"].mean():.4f}')

 year  n_train  logit_no_b5_ll  logit_no_b5_acc  cb_no_b5_ll  cb_no_b5_acc  logit_+b5_ll  logit_+b5_acc  cb_+b5_ll  cb_+b5_acc
 2014       64          0.9904           0.5781       1.0611        0.5000        1.0129         0.5156     1.1520      0.4531
 2018      128          0.9938           0.5469       1.2043        0.5469        1.0632         0.5312     1.2419      0.5156
 2022      192          1.0263           0.5781       1.1147        0.5156        1.0278         0.5781     1.1274      0.5312

  logit_no_b5    LL=1.0035  acc=0.5677
  logit_+b5      LL=1.0346  acc=0.5417
  cb_no_b5       LL=1.1267  acc=0.5208
  cb_+b5         LL=1.1738  acc=0.5000


## Days-of-rest + recent-form features

Unlike big5/manager/injuries, these are computed for EVERY match (not just WC), so the model can learn coefficients properly. Hypothesis: form captures momentum Elo lags on; rest captures fatigue.

In [11]:
from src.form_features import add_form_features

# Compute form features on FULL match history then merge into df
form_df = add_form_features(df_all.sort_values('date').reset_index(drop=True), rolling_window=10)
form_cols = ['home_days_rest','away_days_rest',
             'home_form_wins_l10','away_form_wins_l10',
             'home_form_gd_l10','away_form_gd_l10']
df3 = df_sv.merge(elo_enriched[['date','home_team','away_team','home_elo_pre','away_elo_pre']],
                   on=['date','home_team','away_team'], how='left').dropna(subset=['home_elo_pre','away_elo_pre'])
df3 = df3.merge(form_df[['date','home_team','away_team']+form_cols], on=['date','home_team','away_team'], how='left')
df3['neutral'] = df3['neutral'].fillna(False)
df3['eff_elo_diff']  = (df3['home_elo_pre'] + (~df3['neutral']).astype(int)*HOME_ADVANTAGE) - df3['away_elo_pre']
df3['elo_diff_norm'] = df3['eff_elo_diff']/400.0
df3['log_value_diff']    = np.log10(df3['home_top_n_value_eur'].clip(lower=FLOOR)/df3['away_top_n_value_eur'].clip(lower=FLOOR))
df3['outfield_age_diff'] = df3['home_outfield_mean_age'] - df3['away_outfield_mean_age']
df3['top1_share_diff']   = df3['home_top1_share'] - df3['away_top1_share']
df3['fifa_points_diff']  = (df3['home_fifa_points'].fillna(0) - df3['away_fifa_points'].fillna(0))/1000.0
df3['days_rest_diff']    = (df3['home_days_rest'] - df3['away_days_rest']) / 30.0   # months
df3['form_wins_diff']    = (df3['home_form_wins_l10'] - df3['away_form_wins_l10']) / 10.0
df3['form_gd_diff']      = (df3['home_form_gd_l10']   - df3['away_form_gd_l10'])   / 30.0
df3['outcome']   = df3.apply(lambda r: match_outcome(r['home_score'], r['away_score']), axis=1)
FEAT_CLF_F = FEAT_CLF + ['days_rest_diff','form_wins_diff','form_gd_diff']
valid3 = df3.dropna(subset=FEAT_CLF_F+['outcome'])
valid3 = valid3[(valid3['home_top_n_value_eur']>FLOOR)&(valid3['away_top_n_value_eur']>FLOOR)].copy()
print(f'Valid: {len(valid3):,}')
print(valid3[FEAT_CLF_F].describe().round(3))

Valid: 7,641
       elo_diff_norm  log_value_diff  outfield_age_diff  top1_share_diff  \
count       7641.000        7641.000           7641.000         7641.000   
mean           0.207           0.047             -0.051           -0.009   
std            0.434           1.099              2.624            0.289   
min           -1.601          -3.751            -12.233           -0.925   
25%           -0.075          -0.666             -1.585           -0.167   
50%            0.195           0.048             -0.043           -0.008   
75%            0.479           0.771              1.514            0.148   
max            2.077           3.840             12.078            0.926   

       fifa_points_diff  days_rest_diff  form_wins_diff  form_gd_diff  
count          7641.000        7641.000        7641.000      7641.000  
mean              0.013          -0.015           0.005         0.006  
std               0.374           1.882           0.275         0.487  
min           

In [12]:
# Backtest with +form features
preds_f = {}
for year in [2010, 2014, 2018, 2022]:
    train = valid3[valid3['date'].dt.year < year]
    test  = valid3[(valid3['date'].dt.year == year) & (valid3['tournament']=='FIFA World Cup')]
    if test.empty: continue
    y_train = train['outcome'].to_numpy(); y_test = test['outcome'].to_numpy()
    Xtr = train[FEAT_CLF_F].to_numpy(); Xte = test[FEAT_CLF_F].to_numpy()
    sc = StandardScaler().fit(Xtr)
    p_mlp = proba_ord(fit_mlp(sc.transform(Xtr), y_train), sc.transform(Xte))
    p_xgb = proba_ord(fit_xgb(Xtr, y_train), Xte)
    p_cb  = proba_ord(fit_cb(Xtr, y_train), Xte)
    preds_f[year] = {'y': y_test, 'mlp': p_mlp, 'xgb': p_xgb, 'cb': p_cb}

rows = []
for name in ['mlp','xgb','cb']:
    y = np.concatenate([d['y'] for d in preds_f.values()])
    p = np.vstack([d[name] for d in preds_f.values()])
    s = score(y, p); rows.append({'model': name+'_f', **s})
for combo in [('mlp','xgb','cb'),]:
    ys, ps = [], []
    for d in preds_f.values():
        avg = np.mean([d[m] for m in combo], axis=0)
        ps.append(avg/avg.sum(axis=1, keepdims=True)); ys.append(d['y'])
    s = score(np.concatenate(ys), np.vstack(ps))
    rows.append({'model': 'avg(mlp+xgb+cb)_f', **s})
print('With +days_rest_diff +form_wins_diff +form_gd_diff:')
print(pd.DataFrame(rows).round(4).to_string(index=False))

With +days_rest_diff +form_wins_diff +form_gd_diff:
            model  log_loss  accuracy  brier
            mlp_f    1.0060    0.5352 0.5955
            xgb_f    1.0207    0.5195 0.6048
             cb_f    0.9963    0.5391 0.5920
avg(mlp+xgb+cb)_f    0.9987    0.5469 0.5922


In [13]:
# XGBoost importance with form features
last_train = valid3[valid3['date'].dt.year < 2022]
clf = fit_xgb(last_train[FEAT_CLF_F].to_numpy(), last_train['outcome'].to_numpy())
imp = pd.Series(clf.feature_importances_, index=FEAT_CLF_F).sort_values(ascending=False)
print('XGB gain importance:')
print(imp.round(3).to_string())

XGB gain importance:
elo_diff_norm        0.298
fifa_points_diff     0.171
log_value_diff       0.146
form_gd_diff         0.086
top1_share_diff      0.079
outfield_age_diff    0.075
form_wins_diff       0.074
days_rest_diff       0.071
